In [3]:
%pip install torch transformers peft accelerate datasets bert-score pandas tqdm tabulate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
# -*- coding: utf-8 -*-
"""
(수정-v2) 베이스 모델과 파인튜닝된 모델의 답변 품질을 정성적으로 비교 평가하기 위한 스크립트.

변경 사항:
- 최종 목표인 '새로운 디자인 생성' 능력에 초점을 맞춤.
- 창의적인 컨셉을 제시하고, 그에 맞는 구체적인 디자인 요소를 '생성'하도록 유도하는 질문으로 구성.
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import pandas as pd
from tqdm import tqdm

# --- 설정 ---
BASE_MODEL_PATH = "./kanana1_5_8b_instruct_2505"
ADAPTER_PATH = "./kanana_finetuned_model"
OUTPUT_MD_PATH = "qualitative_comparison_kanana.md"

# 새로운 디자인을 '생성'하는 능력을 평가하기 위한 질문 목록
QUESTIONS = [
    # --- 컨셉 기반 신규 디자인 생성 ---
    "'한국의 전통적인 한옥과 수묵화'를 컨셉으로 제네시스 G90의 새로운 스페셜 에디션 모델을 디자인해줘. 특히 크레스트 그릴, 휠, 실내 내장재의 디자인이 어떻게 바뀔지 구체적으로 묘사해줘.",
    "2050년 미래 해양 도시를 탐험하기 위한 '현대 포세이돈'이라는 이름의 수륙양용 SUV를 상상해서 디자인해줘. 공기역학적인 차체, 잠수 모드를 위한 헤드라이트, 물 속 추진을 위한 휠의 변형 디자인을 중심으로.",
    
    # --- 특정 디자인 요소의 창의적 융합 및 재해석 ---
    "현대자동차의 '파라메트릭 픽셀'과 제네시스의 '두 줄' 디자인을 융합해서, 새로운 전기 스포츠카의 테일램프 디자인을 만들어줘. 어떤 모양일지 아주 상세하게 설명해줘.",
    
    # --- 브랜드 아이덴티티 기반의 새로운 모델 생성 ---
    "현대의 고성능 'N' 브랜드에서 최초의 오프로드용 픽업트럭을 만든다면 어떤 모습일까? 'N' 브랜드의 상징색, 공격적인 범퍼 디자인, 그리고 거친 지형을 위한 타이어와 휠 디자인을 구체적으로 설명해줘."
]

def load_model(model_path, adapter_path=None):
    """모델과 토크나이저를 로드하는 통합 함수"""
    print(f"모델 로드 중: {model_path}" + (f" + {adapter_path}" if adapter_path else ""))
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        trust_remote_code=True,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    if adapter_path:
        model = PeftModel.from_pretrained(model, adapter_path)
        model = model.merge_and_unload()
        print("✅ 파인튜닝 모델 로드 및 병합 완료")
    else:
        print("✅ 베이스 모델 로드 완료")
        
    return model, tokenizer

def generate_answer(model, tokenizer, question):
    """주어진 모델과 질문으로 답변을 생성하는 함수"""
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    try:
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
        
        # 창의적이고 상세한 생성을 위해 옵션 유지
        output = model.generate(
            input_ids, 
            max_new_tokens=512, 
            do_sample=True, 
            temperature=0.8,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1 # 반복을 줄여 좀 더 창의적인 결과 유도
        )
        
        full_text = tokenizer.decode(output[0], skip_special_tokens=True)
        answer = full_text.split("assistant\n")[1].strip() if "assistant\n" in full_text else full_text

        answer = answer.replace("\n", "<br>").replace("|", "&#124;")
        return answer
    except Exception as e:
        return f"답변 생성 중 오류 발생: {str(e)}"

def main():
    """메인 실행 함수"""
    base_model, tokenizer = load_model(BASE_MODEL_PATH)
    base_answers = []
    print("\n--- 베이스 모델 답변 생성 시작 ---")
    for q in tqdm(QUESTIONS, desc="베이스 모델"):
        base_answers.append(generate_answer(base_model, tokenizer, q))
    del base_model
    torch.cuda.empty_cache()

    ft_model, tokenizer = load_model(BASE_MODEL_PATH, ADAPTER_PATH)
    ft_answers = []
    print("\n--- 파인튜닝 모델 답변 생성 시작 ---")
    for q in tqdm(QUESTIONS, desc="파인튜닝 모델"):
        ft_answers.append(generate_answer(ft_model, tokenizer, q))
    del ft_model
    torch.cuda.empty_cache()

    print(f"\n--- 결과를 {OUTPUT_MD_PATH} 파일로 저장 중 ---")
    with open(OUTPUT_MD_PATH, 'w', encoding='utf-8') as f:
        f.write("# 모델별 답변 정성 평가 (v2 - 창의적 디자인 생성 중심)\n\n")
        f.write("| 질문 (Creative Brief) | 베이스 모델 답변 (Base Model) | 파인튜닝 모델 답변 (Finetuned Model) |\n")
        f.write("|---|---|---|")
        for i in range(len(QUESTIONS)):
            f.write(f"| {QUESTIONS[i]} | {base_answers[i]} | {ft_answers[i]} |\n")

    print(f"✅ 정성 평가용 데이터 생성이 완료되었습니다. '{OUTPUT_MD_PATH}' 파일을 확인해주세요.")

if __name__ == "__main__":
    main()


모델 로드 중: ./kanana1_5_8b_instruct_2505


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ 베이스 모델 로드 완료

--- 베이스 모델 답변 생성 시작 ---


베이스 모델: 100%|██████████| 4/4 [00:46<00:00, 11.67s/it]


모델 로드 중: ./kanana1_5_8b_instruct_2505 + ./kanana_finetuned_model


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ 파인튜닝 모델 로드 및 병합 완료

--- 파인튜닝 모델 답변 생성 시작 ---


파인튜닝 모델: 100%|██████████| 4/4 [00:37<00:00,  9.34s/it]


--- 결과를 qualitative_comparison_kanana.md 파일로 저장 중 ---
✅ 정성 평가용 데이터 생성이 완료되었습니다. 'qualitative_comparison_kanana.md' 파일을 확인해주세요.


In [6]:
# -*- coding: utf-8 -*-
"""
(수정-v4) 정량 평가 스크립트.

변경 사항:
- 평가 시 'context'를 프롬프트에 포함하여 훈련-추론 환경을 일치시킴.
- F1-Score 외에 Precision, Recall 점수를 모두 결과에 포함하여 다각적인 분석이 가능하도록 함.
- 최종 보고서에 세 가지 지표(P, R, F1)를 모두 나란히 비교하여 모델의 성향을 파악하기 용이하게 함.
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import json
import bert_score
import pandas as pd
from tqdm import tqdm

# --- 공통 설정 ---
BASE_MODEL_PATH = "./kanana1_5_8b_instruct_2505"
ADAPTER_PATH = "./kanana_finetuned_model"
TEST_DATA_PATH = './test.jsonl'
OUTPUT_MD_PATH = "./quantitative_evaluation_kanana.md"

# 훈련 시 사용한 시스템 프롬프트와 동일하게 정의
SYSTEM_PROMPT = (
    "당신은 자동차 디자인 트렌드와 역사에 정통한 '자동차 디자인 전문 AI'입니다. "
    "특히 현대자동차의 디자인 철학인 '센슈어스 스포티니스'와 '플루이딕 스컬프처'를 깊이 이해하고 있습니다. "
    "사용자의 질문에 대해, 전문 지식을 바탕으로 시각적이고 창의적인 관점에서 상세하게 설명해주세요."
)

def evaluate_model(model_name, model, tokenizer, data):
    """주어진 모델에 대해 평가를 수행하고 결과 데이터프레임을 반환하는 함수"""
    print(f"\n--- [{model_name}] 모델 답변 생성 시작 ---")
    
    predictions, references, questions = [], [], []

    for item in tqdm(data, desc=f"평가 중 [{model_name}]"):
        question = item['messages'][0]['content']
        reference_answer = item['messages'][1]['content']
        context = item.get("context", None) # context 필드 가져오기

        # 훈련 시와 동일한 프롬프트 형식으로 messages 재구성
        full_messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        if context:
            full_messages.append({"role": "system", "content": f"다음은 참고 문맥입니다:\n{context}"})
        full_messages += [{"role": "user", "content": question}]
        
        # apply_chat_template에 재구성된 messages를 전달
        prompt = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=True)
        
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
        
        output = model.generate(
            input_ids, max_new_tokens=256, do_sample=False, eos_token_id=tokenizer.eos_token_id
        )
        
        full_text = tokenizer.decode(output[0], skip_special_tokens=True)
        answer = full_text.split("assistant\n")[1].strip() if "assistant\n" in full_text else full_text

        predictions.append(answer)
        references.append(reference_answer)
        questions.append(question)

    print(f"--- [{model_name}] BERTScore 계산 중 ---")
    bert_p, bert_r, bert_f1 = bert_score.score(
        predictions, references, lang="ko", model_type="bert-base-multilingual-cased", verbose=False
    )
    
    df = pd.DataFrame({
        "Question": questions,
        f"{model_name}_Answer": predictions,
        f"{model_name}_Precision": bert_p.tolist(),
        f"{model_name}_Recall": bert_r.tolist(),
        f"{model_name}_F1_Score": bert_f1.tolist(),
        "Reference Answer": references
    })
    print(f"✅ [{model_name}] 평가 완료")
    return df

def main():
    """메인 실행 함수"""
    with open(TEST_DATA_PATH, 'r', encoding='utf-8') as f:
        test_data = [json.loads(line) for line in f if line.strip()]
    print(f"✅ 총 {len(test_data)}개의 평가 데이터를 로드했습니다.")

    # --- 모델 평가 ---
    base_model, tokenizer = AutoModelForCausalLM.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True, device_map="auto", torch_dtype=torch.bfloat16), AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
    base_results_df = evaluate_model("Base", base_model, tokenizer, test_data)
    del base_model
    torch.cuda.empty_cache()

    base_model_for_peft = AutoModelForCausalLM.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True, device_map="auto", torch_dtype=torch.bfloat16)
    finetuned_model = PeftModel.from_pretrained(base_model_for_peft, ADAPTER_PATH).merge_and_unload()
    ft_results_df = evaluate_model("Finetuned", finetuned_model, tokenizer, test_data)
    del base_model_for_peft, finetuned_model
    torch.cuda.empty_cache()

    # --- 결과 병합 및 저장 ---
    comparison_df = pd.merge(base_results_df, ft_results_df, on=["Question", "Reference Answer"])
    
    # 컬럼 순서 재정렬
    comparison_df = comparison_df[[
        "Question", "Reference Answer", 
        "Base_Answer", "Base_Precision", "Base_Recall", "Base_F1_Score",
        "Finetuned_Answer", "Finetuned_Precision", "Finetuned_Recall", "Finetuned_F1_Score"
    ]]

    # 전체 평균 점수 계산
    avg_scores_text = "| Metric | Base Model | Finetuned Model |\n"
    avg_scores_text += "|---|---|---|"
    for metric in ["Precision", "Recall", "F1_Score"]:
        base_avg = comparison_df[f'Base_{metric}'].mean()
        ft_avg = comparison_df[f'Finetuned_{metric}'].mean()
        avg_scores_text += f"| **{metric}** | {base_avg:.4f} | {ft_avg:.4f} |\n"

    print("\n--- 전체 평균 점수 ---")
    print(avg_scores_text)

    # Markdown 파일로 저장
    with open(OUTPUT_MD_PATH, 'w', encoding='utf-8') as f:
        f.write("# 정량 평가 비교 (P, R, F1) (v2)\n\n")
        f.write("## 전체 평균 점수\n\n")
        f.write(avg_scores_text)
        f.write("\n\n## 개별 결과 비교\n\n")
        f.write(comparison_df.to_markdown(index=False))

    print(f"\n✅ 비교 평가 결과가 '{OUTPUT_MD_PATH}' 파일에 저장되었습니다.")

if __name__ == "__main__":
    main()

✅ 총 40개의 평가 데이터를 로드했습니다.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


--- [Base] 모델 답변 생성 시작 ---


평가 중 [Base]: 100%|██████████| 40/40 [03:19<00:00,  4.99s/it]


--- [Base] BERTScore 계산 중 ---
✅ [Base] 평가 완료


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


--- [Finetuned] 모델 답변 생성 시작 ---


평가 중 [Finetuned]: 100%|██████████| 40/40 [01:14<00:00,  1.87s/it]


--- [Finetuned] BERTScore 계산 중 ---
✅ [Finetuned] 평가 완료

--- 전체 평균 점수 ---
| Metric | Base Model | Finetuned Model |
|---|---|---|| **Precision** | 0.6926 | 0.8948 |
| **Recall** | 0.7827 | 0.8829 |
| **F1_Score** | 0.7344 | 0.8884 |


✅ 비교 평가 결과가 './quantitative_evaluation_kanana.md' 파일에 저장되었습니다.


In [ ]:
"""
🔎 핵심 요약
1. 전체 평균 점수 비교
	•	Precision:
Base 0.6926 → Finetuned 0.8948 (+20pp 개선)
	•	Recall:
Base 0.7827 → Finetuned 0.8829 (+10pp 개선)
	•	F1 Score:
Base 0.7344 → Finetuned 0.8884 (+15pp 개선)
➡️ 정확도와 재현율 모두 상승하면서 균형 지표인 F1도 크게 향상됨 → 파인튜닝 효과가 상당히 명확하게 드러남.
2. 개별 사례 비교
(1) 신형 싼타페 디자인 철학
	•	Base Answer: 맞는 정보 일부 포함했지만 `"센슈어스 스포티니스"`라는 불필요/혼동 요소 추가
→ Precision 낮음 (0.686)
	•	Finetuned Answer: 레퍼런스와 거의 동일한 핵심만 제공
→ Precision/Recall/F1 모두 0.95+
(2) 더 뉴 팰리세이드 특징
	•	Base Answer: 설명은 꽤 상세했지만, 일부 포인트에서 레퍼런스 대비 과잉 서술
→ Precision 낮음 (0.651)
	•	Finetuned Answer: 레퍼런스 핵심을 모두 포함하면서 간결하고 정확
→ Precision/Recall/F1 ~0.86
(3) 현대 HED-5 컨셉트 공개 행사
	•	Base Answer: “제네바 모터쇼”라는 핵심은 맞지만 “세계 최초” 설명 등은 누락 → Recall 높지만 Precision 제한적
	•	Finetuned Answer: “제네바 모터쇼 세계 최초 공개”라는 정확한 언급 + 맥락 설명
→ F1 점수 안정적 상승
📌 종합 인사이트
	1.	Base 모델은 설명이 길고 풍부하지만 → 불필요 요소나 혼동되는 표현 때문에 Precision 낮음.
	2.	Finetuned 모델은 레퍼런스 기준 정확하고 간결하게 답변하여 Precision 대폭 개선.
	3.	Recall 역시 Base 대비 안정적으로 유지되거나 오히려 다소 개선됨.
	4.	따라서 파인튜닝은 ‘불필요한 추가 서술 최소화 + 레퍼런스 정답과 높은 일치율 확보’에 효과적임을 보여줌.
"""

In [4]:
%pip install tabulate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
